# 08 — External-HD vs locked-TEST gap, depth QC, TENX128/TENX155 checks (Supplement S5)
Descriptive only: nothing is selected or tuned, no p-values. Inputs are the per-slide scenario records (`res17_per_slide_mask_seed.csv`, regenerated by notebook 04 if present, else the stored revision file), the slide→cancer map and the spatial nearest-neighbour audit.
Regenerated tables are compared with `revision_resolution_2026-10-21/p8_*.csv`. The depth-QC and TENX128/155 outputs have **no stored CSV** (they were printed in the revision); they are printed here for comparison with the manuscript numbers. Needs the raw `.h5ad` files.

In [ ]:
# --- Setup: mount Drive, unpack the repository, install dependencies ---
from google.colab import drive
drive.mount('/content/drive')
import subprocess, sys, os
BASE = '/content/drive/MyDrive/New-SASP-idea'                 # data/checkpoints live here (not in the repo)
REPO_ZIP = BASE + '/code_release_export_2026-10-23/spatial-sasp-gat.zip'   # or: git clone <repo URL> /content/spatial-sasp-gat
import shutil, hashlib
if os.path.isdir('/content/spatial-sasp-gat'):
    shutil.rmtree('/content/spatial-sasp-gat')          # never reuse an older unpacked copy: always run the zip that is on Drive
subprocess.run(['unzip', '-q', '-o', REPO_ZIP, '-d', '/content'], check=True)
print('repository zip sha256[:16]:', hashlib.sha256(open(REPO_ZIP, 'rb').read()).hexdigest()[:16], '| size', os.path.getsize(REPO_ZIP))
if 'sasp_gat' in sys.modules: print('WARNING: sasp_gat was already imported in this runtime - reset the runtime so the new code is used')
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch_geometric'], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'], check=True)   # reads HEST .h5ad; scikit-learn / networkx are preinstalled on Colab (do NOT reinstall: it can swap pandas under a running kernel)
# data_tables exported in ENV-2/ENV-3 (copied only if missing in the repo)
import shutil
for f in ['sasp54_genes_canonical_order.csv', 'ppi_string_v12_edges.csv', 'slide_partition.csv']:
    dst = '/content/spatial-sasp-gat/data_tables/' + f
    if not os.path.exists(dst):
        shutil.copy(BASE + '/code_release_export_2026-10-23/' + f, dst)
sys.path.insert(0, '/content/spatial-sasp-gat/src')
import torch, torch_geometric
print('torch', torch.__version__, '| torch_geometric', torch_geometric.__version__,
      '| cuda', torch.cuda.is_available())

In [ ]:
import hashlib, json, pandas as pd, numpy as np, torch
from pathlib import Path
from sasp_gat.config import *
from sasp_gat.data import load_package, split_samples, sha256_file
from sasp_gat.ppi import ppi_edge_index_from_networkx
from sasp_gat.availability import build_availability, effective_mask, score_mask
from sasp_gat.evaluation import load_model, eval_both
from sasp_gat.training import evaluate_multimask_av

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
REPO = Path('/content/spatial-sasp-gat'); T = REPO / 'data_tables'
PKL = Path(BASE) / 'MODEL_READY_DATASET_v1_2026-09-26.pkl'
pkg, pyg_dataset, gene_order = load_package(PKL)
train_s, val_s, test_s, hd_s = split_samples(pyg_dataset)
AVAIL, LOGGED = build_availability(pyg_dataset.keys(), gene_order, T / 'availability_unmeasured.csv')
ppi_ei = ppi_edge_index_from_networkx(pkg['ppi_graph_networkx'], gene_order, device)
CKDIR = REPO / 'checkpoints' if (REPO / 'checkpoints' / 'full_avail_seed42.pt').exists() else Path(BASE) / 'revision_checkpoints_2026-10-18'
print('checkpoint dir:', CKDIR)
import torch_geometric, scipy; print('versions: torch', torch.__version__, '| pyg', torch_geometric.__version__, '| numpy', np.__version__, '| pandas', pd.__version__, '| scipy', scipy.__version__)
import warnings; warnings.filterwarnings('ignore')
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'anndata'])   # only needed to read HEST .h5ad coordinates
from sasp_gat.training import vkey
from sasp_gat.baselines import (train_gene_means, legacy_gene_means, eval_baseline, build_neighbour_edges,
                                eval_neighbour_methods)
ABL_DIR = Path(BASE) / 'revision_checkpoints_2026-10-18'
TAB_DIR = Path(BASE) / 'revision_tables_2026-10-18'
BL_DIR  = Path(BASE) / 'revision_spatial_baselines_2026-10-18'
VARIANTS = ['expression-only', 'expression+spatial', 'expression+PPI']
ms = lambda s: f'{s.mean():.4f} ± {s.std(ddof=1):.4f}'
import warnings; warnings.filterwarnings('ignore')
from sasp_gat.verify import compare_csv, report_compare
MR_DIR = Path(BASE) / 'revision_masking_robustness_2026-10-19'
OUT_DIR = Path(BASE) / 'repro_outputs_2026-10-24'; OUT_DIR.mkdir(exist_ok=True)
H5_ROOT = Path(BASE) / 'hest_expression'; H5_DIR = H5_ROOT / 'st' if (H5_ROOT / 'st').exists() else H5_ROOT

from sasp_gat import hd_gap as hg
f_res = OUT_DIR / 'res17_per_slide_mask_seed.csv'
RES = pd.read_csv(f_res if f_res.exists() else MR_DIR / 'res17_per_slide_mask_seed.csv', float_precision='round_trip')
print('RES17 from', 'regenerated (notebook 04)' if f_res.exists() else 'STORED revision file', '| rows', len(RES))
P8 = Path(BASE) / 'revision_resolution_2026-10-21'
cmap = pd.read_csv(Path(BASE) / 'revision_audit_2026-10-19' / 'slide_cancer_map.csv')
nn = pd.read_csv(Path(BASE) / 'checkpoint_2026-09-25' / 'spatial_NN_distance_audit_2026-09-26.csv', float_precision='round_trip')


In [ ]:
# A) slide-level and pooled tables (rand25) + geometry consistency, vs stored p8_* files
W, POOL, SKILL, shared = hg.slide_and_pooled_tables(RES, cmap)
slides = sorted(W.slide)
GEO = hg.geometry_table(H5_DIR, slides, cmap, nn, RES)
for n, d in [('p8_slide_level_rand25_v1.csv', W), ('p8_cohort_cancer_pooled_mse_rand25_v1.csv', POOL),
             ('p8_cohort_cancer_pooled_skill_rand25_v1.csv', SKILL), ('p8_geometry_consistency_v1.csv', GEO)]:
    d.to_csv(OUT_DIR / n, index=False)
report_compare({n: compare_csv(d, P8 / n, rtol=1e-6, atol=1e-9) for n, d in
                [('p8_slide_level_rand25_v1.csv', W), ('p8_cohort_cancer_pooled_mse_rand25_v1.csv', POOL),
                 ('p8_cohort_cancer_pooled_skill_rand25_v1.csv', SKILL), ('p8_geometry_consistency_v1.csv', GEO)]})
print('cancer types shared by TEST and HD:', shared)
print(POOL[POOL.level != 'kanser türü'].round(5).to_string(index=False))

In [ ]:
# B) depth QC (raw counts of the h5ad vs slide MSE) -- printed only; no stored CSV
T0 = hg.slide_mse_table(RES)
T0 = T0[T0.cohort.isin(['test', 'hd'])].reset_index(drop=True)
TQ, COR = hg.depth_qc(T0, H5_DIR, pyg_dataset)
TQ.to_csv(OUT_DIR / 'qc_depth_slide.csv', index=False); COR.to_csv(OUT_DIR / 'qc_depth_spearman.csv', index=False)
print(COR.round(3).to_string(index=False))
print(TQ.groupby('cohort')[['med_total_counts', 'med_genes_detected', 'n_spots']].median().round(1))

In [ ]:
# C) TENX128 / TENX155 barcode-aligned comparison and sensitivity of HD results (no retraining)
OV = hg.tenx_overlap(H5_DIR); print('TENX128 vs TENX155 (compared by count vector, not by barcode):'); print(pd.Series(OV).to_string())
pd.Series(OV).to_csv(OUT_DIR / 'tenx128_vs_tenx155_overlap.csv', header=['value'])
RALL = RES.copy()
f2 = Path(BASE) / 'revision_2hop_2026-10-23' / 'res_2hop_per_slide_mask_seed.csv'
if f2.exists(): RALL = pd.concat([RALL, pd.read_csv(f2)], ignore_index=True)
else: print('2-hop file not found; sensitivity covers the scenarios of res17 only')
SENS = hg.tenx_sensitivity(RALL)
for scn, t in SENS.items():
    print('\n', scn); print((t * 1000).round(2).to_string())
    t.to_csv(OUT_DIR / f'tenx_sensitivity_{scn}.csv')
print('\nCompare these numbers with the manuscript / response-letter HD-sensitivity statement (MSE x 1e-3).')

## Exact duplicate count vectors across all 76 slides
Compares every spot's (gene index, count) vector between all slides (spots with >= 50 nonzero genes). Finding of the review: only TENX128 and TENX155 (both external HD) share vectors (about 98% of rows, under different barcodes and coordinates); no pair of slides from different splits shares any. Only exact copies are detected, not rescaled or altered ones.

In [ ]:
split = {s: pyg_dataset[s].split for s in pyg_dataset}
files_all = {p.stem: p for p in H5_ROOT.rglob('*.h5ad')}
DUP = hg.row_vector_audit(files_all, sorted(pyg_dataset), split)
print('slide pairs sharing at least one identical count vector:', len(DUP)); print(DUP.round(4).to_string(index=False))
cross = DUP[DUP.split_a != DUP.split_b]; print('pairs from different splits:', 'none' if len(cross) == 0 else cross.to_string(index=False))
DUP.to_csv(OUT_DIR / 'duplicate_count_vectors_across_slides.csv', index=False)